# LegalCheck India — Notebook 2: Pipeline 2 (pre-trained text generation, no fine-tuning)
**Group 10 · ISOM5240 · Gollakaram Vishnu Srivardhan (21401910, leader) & Rahul Singh (2610gnam02)**

When Pipeline 1 says a narrative is `VALID`, Pipeline 2 writes the body of a **preliminary complaint** (facts, grounds, relief) that Python wraps in a fixed template with `__ [Field]` placeholders for missing details.

**Three comparable small instruction models** (all run on Streamlit Cloud CPU):
`Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen3-0.6B` (thinking disabled), `HuggingFaceTB/SmolLM2-360M-Instruct`.

**Accuracy** = share of 12 drafting cases whose draft passes all four objective checks
(key facts kept ≥ 75 %, required headings + numbered paragraphs, no invented section numbers/case law, clean length).
**Runtime** = mean seconds per draft after a warm-up run (warm-up excluded from timing, never from accuracy).
Greedy decoding (`do_sample=False`) makes the results repeatable.

In [ ]:
# Step 0 - Install packages
!pip install -q "transformers>=4.51.0,<5.0.0" "accelerate>=0.34.0" pandas

In [ ]:
# ---- Hardware and library versions (recorded for reproducibility) ----
import platform, torch, transformers
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")

In [ ]:
# ---- Get the project files (dataset CSVs) ----
# Works three ways, all using the SAME frozen CSV files:
#   (1) CSVs uploaded anywhere under /content (or the submitted data folder) are found automatically;
#   (2) an uploaded Colab_pack.zip / GitHub_repo_files.zip is unzipped automatically;
#   (3) otherwise the public GitHub repository (REPO_URL) is cloned.
import glob, os, subprocess, zipfile
REPO_URL = "https://github.com/vishnusrivardhangollakaram-boop/Group10_DLBAWP_Group_Project.git"
SEARCH_ROOTS = [os.getcwd(), "/content", ".."]


def find_data_folder():
    """Return the first folder (searched recursively) that contains train.csv, or None."""
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in ("sample_data", "drive", ".config", "checkpoints")]
            if "train.csv" in filenames and "validation.csv" in filenames:
                return dirpath
    return None


DATA_FOLDER = find_data_folder()
if DATA_FOLDER is None:                                   # (2) unzip an uploaded pack if present
    for archive in glob.glob("/content/*.zip") + glob.glob("*.zip"):
        with zipfile.ZipFile(archive) as handle:
            handle.extractall("/content/unzipped_data")
    DATA_FOLDER = find_data_folder()
if DATA_FOLDER is None:                                   # (3) clone the public repository
    if "OWNER/REPO" in REPO_URL:
        raise FileNotFoundError("Upload train.csv, validation.csv, test.csv, app_test_cases.csv and "
                                "drafting_eval_cases.csv to the Colab Files panel, then run this cell again.")
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    DATA_FOLDER = find_data_folder()
print(f"Using data folder: {DATA_FOLDER}")
print(sorted(f for f in os.listdir(DATA_FOLDER) if f.endswith(".csv")))

## Step 1 - Shared core logic (identical copy of `app/legalcheck_core.py` used by the Streamlit app)

In [ ]:
"""
LegalCheck India - shared core logic (Group 10, ISOM5240)
=========================================================
One source of truth for the decision rule, the drafting prompt, the complaint template and the
draft quality checks. The Streamlit app imports this file, and the same code is copied verbatim
into the Colab notebooks, so the evaluated logic and the deployed logic are identical.

Pipeline 1 (fine-tuned)  : "text-classification" -> VALID / NOT_VALID legal issue in India
Pipeline 2 (pre-trained) : "text-generation"     -> body of a preliminary complaint (facts, grounds, relief)
Python                   : wraps the body in a fixed complaint template with __ [Field] placeholders
"""

# ==============================
# IMPORT PART
# ==============================
import re
import time

# ---- Settings and constants ----
VALID_LABEL = "VALID"
NOT_VALID_LABEL = "NOT_VALID"
VALIDITY_THRESHOLD = 0.50            # VALID when the model score for VALID is at least this value
MINIMUM_INPUT_WORDS = 8              # shorter inputs are rejected with a helpful message
MAXIMUM_INPUT_WORDS = 350            # longer inputs are rejected (no silent truncation of facts)
CLASSIFIER_MAX_TOKENS = 128          # same max_length as used during fine-tuning
DRAFT_MAX_NEW_TOKENS = 420
DRAFT_REPETITION_PENALTY = 1.10
FACT_RETENTION_PASS_LEVEL = 0.75     # share of key facts that must appear in the draft
DRAFT_MINIMUM_WORDS = 60
DRAFT_MAXIMUM_WORDS = 420

DRAFT_SYSTEM_PROMPT = (
    "You are a careful legal drafting assistant for an Indian law firm. "
    "Use only the facts the client gave. Never invent names, dates, amounts, places, documents, witnesses "
    "or section numbers. If a needed detail is missing, write a placeholder such as __ [Date of incident]. "
    "Write in formal English."
)

DRAFT_USER_TEMPLATE = (
    "Client narrative:\n\"\"\"{narrative}\"\"\"\n\n"
    "Write the body of a preliminary complaint using exactly these three headings:\n"
    "FACTS OF THE CASE: numbered paragraphs (1., 2., 3., ...) restating the facts in the third person "
    "(\"The complainant ...\"), keeping every name, date, place and amount given.\n"
    "GROUNDS: two or three numbered points explaining briefly why this is a legal grievance and its nature "
    "(criminal, civil, consumer, family, labour or cyber).\n"
    "RELIEF SOUGHT: one to three numbered requests.\n"
    "Do not cite any section numbers or case law. Keep it under 300 words."
)


# ==============================
# FUNCTION PART
# ==============================
def count_words(text):
    """Return the number of words in a text."""
    return len(re.findall(pattern=r"\S+", string=text or ""))


def validate_narrative(narrative):
    """Check the user's narrative. Returns (is_ok, message)."""
    word_count = count_words(text=narrative)
    if word_count < MINIMUM_INPUT_WORDS:
        return False, f"Please describe the issue in at least {MINIMUM_INPUT_WORDS} words (you wrote {word_count})."
    if word_count > MAXIMUM_INPUT_WORDS:
        return False, (f"Please shorten the description to {MAXIMUM_INPUT_WORDS} words or fewer "
                       f"(you wrote {word_count}); facts are never cut silently.")
    return True, "ok"


def decide_validity(classifier_output):
    """Turn text-classification output (list of {label, score}) into (decision_label, valid_score)."""
    scores = {item["label"]: float(item["score"]) for item in classifier_output}
    valid_score = scores.get(VALID_LABEL, 1.0 - scores.get(NOT_VALID_LABEL, 0.0))
    decision_label = VALID_LABEL if valid_score >= VALIDITY_THRESHOLD else NOT_VALID_LABEL
    return decision_label, valid_score


def classify_narrative(classifier, narrative):
    """Run Pipeline 1 on one narrative. Returns (decision_label, valid_score, seconds)."""
    start_time = time.perf_counter()
    classifier_output = classifier(narrative, top_k=None, truncation=True, max_length=CLASSIFIER_MAX_TOKENS)
    if classifier_output and isinstance(classifier_output[0], list):      # some versions nest the list
        classifier_output = classifier_output[0]
    decision_label, valid_score = decide_validity(classifier_output=classifier_output)
    return decision_label, valid_score, time.perf_counter() - start_time


def build_draft_prompt(tokenizer, narrative):
    """Build the chat prompt string for Pipeline 2 (thinking disabled for Qwen3; ignored by other models)."""
    messages = [
        {"role": "system", "content": DRAFT_SYSTEM_PROMPT},
        {"role": "user", "content": DRAFT_USER_TEMPLATE.format(narrative=narrative.strip())},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)


def clean_generated_body(raw_text):
    """Remove empty thinking blocks, markdown symbols and extra blank lines from the generated body."""
    text = re.sub(pattern=r"<think>.*?</think>", repl="", string=raw_text or "", flags=re.DOTALL)
    text = text.replace("**", "").replace("##", "").replace("```", "")
    text = re.sub(pattern=r"\n{3,}", repl="\n\n", string=text)
    return text.strip()


def generate_draft_body(generator, narrative, streamer=None):
    """Run Pipeline 2 on one narrative. Returns (body_text, seconds)."""
    prompt = build_draft_prompt(tokenizer=generator.tokenizer, narrative=narrative)
    start_time = time.perf_counter()
    generation_kwargs = dict(max_new_tokens=DRAFT_MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=DRAFT_REPETITION_PENALTY, return_full_text=False)
    if streamer is not None:
        generation_kwargs["streamer"] = streamer
    output = generator(prompt, **generation_kwargs)
    body_text = clean_generated_body(raw_text=output[0]["generated_text"])
    return body_text, time.perf_counter() - start_time


def assemble_complaint(body_text, narrative_subject="__ [Short subject of the complaint]"):
    """Wrap the generated body in the fixed preliminary-complaint template (Python controls the layout)."""
    header = (
        "PRELIMINARY COMPLAINT (DRAFT FOR ADVOCATE REVIEW)\n\n"
        "BEFORE: __ [Court / Police Station / Consumer Forum name], __ [Place]\n\n"
        "COMPLAINANT: __ [Complainant full name], __ [Age], __ [Address], __ [Phone]\n"
        "AGAINST: __ [Respondent / accused name], __ [Address]\n\n"
        f"SUBJECT: {narrative_subject}\n\n"
        "Respected Sir / Madam,\n\n"
    )
    footer = (
        "\n\nLIST OF DOCUMENTS: __ [Documents to be attached, e.g. receipts, messages, photos]\n\n"
        "VERIFICATION: I, __ [Complainant full name], state that the facts above are true to the best of my "
        "knowledge and belief.\n\n"
        "Place: __ [Place]        Date: __ [Date]        Signature: __ [Signature]\n\n"
        "Note: Machine-generated preliminary draft. It must be reviewed and completed by an advocate."
    )
    return f"{header}{body_text.strip()}{footer}"


def list_missing_fields(draft_text):
    """Return the unique __ [Field] placeholders still present in a draft (missing-information checklist)."""
    fields = re.findall(pattern=r"__ \[([^\]]+)\]", string=draft_text or "")
    return list(dict.fromkeys(field.strip() for field in fields))


def score_draft_body(body_text, key_facts):
    """Objective rubric for one generated body. Returns a dict of checks and an overall pass flag.

    Checks: (1) key-fact retention >= 75 %, (2) required headings + numbered paragraphs,
            (3) no invented section numbers / case law, (4) clean length and no stray reasoning text.
    """
    lower_body = (body_text or "").lower()
    facts = [fact.strip() for fact in key_facts if fact.strip()]
    retained = sum(1 for fact in facts if fact.lower() in lower_body)
    fact_retention = retained / len(facts) if facts else 1.0
    has_structure = ("facts" in lower_body and "relief" in lower_body
                     and len(re.findall(pattern=r"(?m)^\s*\d+[.)]", string=body_text or "")) >= 2)
    no_invented_law = re.search(pattern=r"\b(section|sec\.|article|u/s)\s*\d+|\bipc\b|\bbns\b|\bcrpc\b", string=lower_body) is None
    word_total = count_words(text=body_text)
    clean_length = (DRAFT_MINIMUM_WORDS <= word_total <= DRAFT_MAXIMUM_WORDS and "<think" not in lower_body
                    and re.search(pattern=r"[一-鿿]", string=body_text or "") is None)
    checks = {
        "fact_retention": round(fact_retention, 2),
        "facts_ok": fact_retention >= FACT_RETENTION_PASS_LEVEL,
        "structure_ok": has_structure,
        "no_invented_law_ok": no_invented_law,
        "clean_length_ok": clean_length,
        "words": word_total,
    }
    checks["pass"] = all([checks["facts_ok"], checks["structure_ok"], checks["no_invented_law_ok"],
                          checks["clean_length_ok"]])
    return checks

In [ ]:
# Step 2 - Settings and evaluation cases
import json
import pandas as pd
from transformers import pipeline

CANDIDATE_MODELS = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen3-0.6B", "HuggingFaceTB/SmolLM2-360M-Instruct"]
RESULTS_FOLDER = "results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)
cases_df = pd.read_csv(os.path.join(DATA_FOLDER, "drafting_eval_cases.csv"))
print(f"{len(cases_df)} drafting cases")
cases_df.head(3)

In [ ]:
# Step 3 - Evaluate one candidate model on all drafting cases
def evaluate_generator(model_name):
    """Load a text-generation pipeline, warm it up, draft every case and score it."""
    load_start = time.perf_counter()
    generator = pipeline("text-generation", model=model_name, device=DEVICE,
                         torch_dtype=torch.float16 if DEVICE == 0 else torch.float32)
    load_seconds = time.perf_counter() - load_start
    generate_draft_body(generator=generator, narrative="My neighbour broke my window with a stone yesterday.")  # warm-up
    rows = []
    for case in cases_df.itertuples():
        body_text, seconds = generate_draft_body(generator=generator, narrative=case.text)
        checks = score_draft_body(body_text=body_text, key_facts=case.key_facts.split("|"))
        rows.append({"model": model_name, "case_id": case.case_id, "seconds": round(seconds, 2), **checks,
                     "draft_body": body_text})
        print(f"{case.case_id}: pass={checks['pass']} facts={checks['fact_retention']:.2f} time={seconds:.2f}s")
    del generator
    torch.cuda.empty_cache() if DEVICE == 0 else None
    return pd.DataFrame(rows), load_seconds

In [ ]:
# Step 4 - Run the three candidates
detail_frames, summaries = [], []
for candidate in CANDIDATE_MODELS:
    print(f"\n========== {candidate} ==========")
    detail, load_seconds = evaluate_generator(model_name=candidate)
    detail_frames.append(detail)
    summaries.append({"model": candidate,
                      "accuracy_pass_rate": round(detail["pass"].mean(), 4),
                      "mean_fact_retention": round(detail["fact_retention"].mean(), 4),
                      "structure_ok_rate": round(detail["structure_ok"].mean(), 4),
                      "no_invented_law_rate": round(detail["no_invented_law_ok"].mean(), 4),
                      "clean_length_rate": round(detail["clean_length_ok"].mean(), 4),
                      "mean_runtime_sec": round(detail["seconds"].mean(), 2),
                      "median_runtime_sec": round(detail["seconds"].median(), 2),
                      "load_sec": round(load_seconds, 2)})
details_df = pd.concat(detail_frames, ignore_index=True)
summary_df = pd.DataFrame(summaries)
details_df.to_csv(f"{RESULTS_FOLDER}/pipeline2_draft_details.csv", index=False)
summary_df.to_csv(f"{RESULTS_FOLDER}/pipeline2_model_comparison.csv", index=False)
summary_df

In [ ]:
# Step 5 - Select the best drafting model: highest pass rate; tie -> higher fact retention -> faster
ranked = summary_df.sort_values(by=["accuracy_pass_rate", "mean_fact_retention", "mean_runtime_sec"],
                                ascending=[False, False, True]).reset_index(drop=True)
BEST_GENERATOR = ranked.loc[0, "model"]
print(ranked[["model", "accuracy_pass_rate", "mean_fact_retention", "mean_runtime_sec"]].to_string(index=False))
print(f"\nSelected drafting model: {BEST_GENERATOR}")
json.dump({"selected_generator": BEST_GENERATOR}, open(f"{RESULTS_FOLDER}/pipeline2_selection.json", "w"))

In [ ]:
# Step 6 - Read one full draft from the selected model (complete template, as shown in the app)
example = details_df.query("model == @BEST_GENERATOR").iloc[0]
complaint = assemble_complaint(body_text=example.draft_body)
print(complaint)
print("\nMissing-information checklist:", list_missing_fields(draft_text=complaint))

In [ ]:
# Step 7 - Package results
!zip -q -r results_notebook2.zip results
print(open(f"{RESULTS_FOLDER}/pipeline2_model_comparison.csv").read())